# A9 and A10. OCR-read label and image quality on Kaggle

Runs `src/ocr_read.py` (A9, EasyOCR on the 7,000 attacked images with no defence) on the GPU, and at the same time `src/image_quality.py` (A10, PSNR and SSIM for the 25,800 defended images) on the CPU.

**Before you start**
1. Settings: Accelerator **GPU T4**, Internet **on**.
2. Add Input: the **pixelguard-full** dataset from A7.
3. Add-ons > Secrets: `GITHUB_TOKEN` switched on.

**How to run**
1. Click **Save Version > Save & Run All (Commit)**. Not Run All, or the files are not kept. About 45 minutes.
2. When it finishes, the version's **Output** tab should list `ocr_read.csv`, `image_quality.csv` and `image_quality_rows.csv`, and the last cell should print `OK`.
3. Download the three files and send them to Mehdi.

In [ ]:
!pip install -q easyocr rapidfuzz
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE, OCR will be very slow, turn on the GPU')

In [ ]:
import os, subprocess
from kaggle_secrets import UserSecretsClient

os.chdir('/kaggle/working')
token = UserSecretsClient().get_secret('GITHUB_TOKEN')
repo = 'github.com/Mehdi-Abidi/VisionLanguageTypographicDefence.git'
subprocess.run(['rm', '-rf', '/tmp/repo'])
r = subprocess.run(['git', 'clone', '-q', f'https://{token}@{repo}', '/tmp/repo'], capture_output=True, text=True)
print('cloned' if r.returncode == 0 else 'clone failed: ' + r.stderr.replace(token, '***'))
subprocess.run(['git', '-C', '/tmp/repo', 'remote', 'set-url', 'origin', f'https://{repo}'])
del token
os.chdir('/tmp/repo')
print(subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout)

In [ ]:
import glob, zipfile
from pathlib import Path
import pandas as pd

manifest = pd.read_csv('manifests/full.csv')
ROOT = Path('/tmp/pg_root')
(ROOT / 'images').mkdir(parents=True, exist_ok=True)
need = {Path(p).name for p in manifest['path']}
loose = {Path(f).name: f for f in glob.glob('/kaggle/input/**/*.png', recursive=True) if Path(f).name in need}
for name, f in loose.items():
    if not (ROOT / 'images' / name).exists():
        (ROOT / 'images' / name).symlink_to(f)
from_zip = 0
for zp in sorted(glob.glob('/kaggle/input/**/images_*.zip', recursive=True)):
    with zipfile.ZipFile(zp) as z:
        for member in z.namelist():
            name = Path(member).name
            if name in need and not (ROOT / 'images' / name).exists():
                (ROOT / 'images' / name).write_bytes(z.read(member))
                from_zip += 1
missing = [p for p in manifest['path'] if not (ROOT / p).exists()]
print(len(manifest), 'manifest rows,', len(loose), 'loose images,', from_zip, 'from zips,', len(missing), 'missing')
assert not missing, 'add the pixelguard-full dataset with Add Input'

In [ ]:
import time
t0 = time.time()
a10 = subprocess.Popen(['python', 'src/image_quality.py', '--images-root', str(ROOT),
                        '--out', '/kaggle/working/image_quality.csv', '--rows', '/kaggle/working/image_quality_rows.csv'],
                       stdout=open('/kaggle/working/a10_log.txt', 'w'), stderr=subprocess.STDOUT)
!python src/ocr_read.py --images-root {ROOT} --out /kaggle/working/ocr_read.csv
print(f'A9 took {(time.time() - t0) / 60:.0f} min, waiting for A10')
a10.wait()
print(f'A10 finished with code {a10.returncode} after {(time.time() - t0) / 60:.0f} min')
print(open('/kaggle/working/a10_log.txt').read()[-1500:])

In [ ]:
import numpy as np

ocr = pd.read_csv('/kaggle/working/ocr_read.csv')
quality = pd.read_csv('/kaggle/working/image_quality.csv')
rows = pd.read_csv('/kaggle/working/image_quality_rows.csv')
attacked = manifest[(manifest['defence'] == 'none') & manifest['cell'].isin(['c_on', 'i_on', 'i_off', 'c_off', 'c_long'])]
defended = manifest[manifest['defence'] != 'none']
checks = {
    'A9 one row per attacked image': len(ocr) == len(attacked) == 7000,
    'A9 no repeats': not ocr.duplicated(['image_id', 'cell', 'setting']).any(),
    'A9 columns': list(ocr.columns) == ['image_id', 'cell', 'setting', 'ocr_read', 'ocr_confidence', 'ocr_text'],
    'A10 ten defences': len(quality) == 10,
    'A10 every defended image': int(quality['n'].sum()) == len(rows) == len(defended) == 25800,
    'A10 PSNR mean finite': bool(np.isfinite(quality['psnr_mean']).all()),
}
print(ocr.pivot_table(index='setting', columns='cell', values='ocr_read').round(2))
print(quality.round(3).to_string(index=False))
print(checks)
print('OK' if all(checks.values()) else 'PROBLEM')